# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS42024/shailesh-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# One row = one page in one calendar month which is March 2026. Each Row Summarizes how that page performed in the search of that Mont. I Chose a mid panel month so that June 2026, the final month, stays sealed as a Test Month.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
df = pd.DataFrame({
    "clicks_last_month": rng.poisson(20, 2000),
    "position": rng.uniform(1, 30, 2000),      # was "positions"
})
df["label"] = (df["clicks_last_month"] + rng.normal(0, 15, 2000) > 25).astype(int)

def score(features):
    X_tr, X_te, y_tr, y_te = train_test_split(
        df[features], df["label"], test_size=0.3, random_state=0)
    m = RandomForestClassifier(random_state=0).fit(X_tr, y_tr)
    return roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])

print("honest:", score(["clicks_last_month", "position"]))

df["leak"] = df["label"] + rng.normal(0, 0.05, 2000)
print("with leak:", score(["clicks_last_month", "position", "leak"]))

honest: 0.5586962667945078
with leak: 1.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [8]:
import pandas as pd

# --- Load your data (reuse the df you already have from earlier cells) ---
# If df isn't already defined in this session, uncomment and set the path:
# df = pd.read_csv("your_file.csv")

print("All columns in your dataset:\n", df.columns.tolist())

# --- Sort every column into exactly one bucket below ---
# Move each column name (as a string) from "columns_to_sort" into the right list.

FEATURE = [
    # e.g. "word_count", "internal_links", "page_age_days",
]

LABEL = [
    # e.g. "ranking_improved",   <-- usually just ONE field goes here
]

CONTEXT = [
    # e.g. "url", "page_category", "date",
]

EXCLUDED = {
    # column_name: "reason for excluding it"
    # e.g. "clicks_after_refresh": "leaks the label — only known after the outcome",
    # e.g. "raw_html": "not usable as a numeric/categorical feature without further processing",
}

# --- Sanity check: make sure every column got sorted somewhere ---
sorted_cols = set(FEATURE) | set(LABEL) | set(CONTEXT) | set(EXCLUDED.keys())
all_cols = set(df.columns)

missing = all_cols - sorted_cols
extra = sorted_cols - all_cols

if missing:
    print("\n⚠️  These columns haven't been sorted into any bucket yet:", missing)
if extra:
    print("\n⚠️  These names don't match any real column (typo?):", extra)
if not missing and not extra:
    print("\n✅ Every column is accounted for.")

print(f"\nFeature: {len(FEATURE)} | Label: {len(LABEL)} | Context: {len(CONTEXT)} | Excluded: {len(EXCLUDED)}")

# --- Print excluded fields with their reasons, for your write-up ---
if EXCLUDED:
    print("\nExcluded fields and reasons:")
    for col, reason in EXCLUDED.items():
        print(f"  - {col}: {reason}")


All columns in your dataset:
 ['clicks_last_month', 'position', 'label', 'leak']

⚠️  These columns haven't been sorted into any bucket yet: {'leak', 'position', 'clicks_last_month', 'label'}

Feature: 0 | Label: 0 | Context: 0 | Excluded: 0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:

#Grain. One row per `<KEY columns>`. The query below shows `<N>` rows and `<N>` distinct keys, with `<0>` duplicate keys, so the claim holds.

#Counts. The dataset has `<N>` rows, which matches the contract's expected size.

#Missing values. The most incomplete column is `<column>` at `<X>%` missing. All fields planned as features have less than `<Y>%` missing, so they are safe to use. `<Column>` is excluded because `<X>%` of its values are missing.

#Window. The data covers `<min date>` to `<max date>`, matching the contract's stated window. `<No months are missing / Month M has no rows>`.


from itertools import combinations

print("rows:", len(df))
print("fully duplicated rows:", df.duplicated().sum())
print(df.nunique().sort_values(ascending=False).head(10))

# Search for the smallest set of columns that uniquely identifies a row
cols_ok = [c for c in df.columns if df[c].isna().mean() < 0.05]
KEY = None
for size in (1, 2, 3):
    for cols in combinations(cols_ok, size):
        if not df.duplicated(subset=list(cols)).any():
            KEY = list(cols)
            break
    if KEY:
        break

if KEY:
    print("\nGRAIN FOUND. One row per:", KEY)
    print("rows:", len(df), "| distinct keys:", df[KEY].drop_duplicates().shape[0])
else:
    # No unique key exists, so report the closest one and how many rows repeat
    best = df.nunique().sort_values(ascending=False).index[0]
    dupes = df.duplicated(subset=[best], keep=False).sum()
    print("\nNO UNIQUE KEY. Closest column:", best)
    print(f"rows sharing a {best} value with another row:", dupes)
    KEY = [best]
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


rows: 2000
fully duplicated rows: 0
position             2000
leak                 2000
clicks_last_month      31
label                   2
dtype: int64

GRAIN FOUND. One row per: ['position']
rows: 2000 | distinct keys: 2000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [12]:
# Unbalanced history: rows per month

# Auto-detect date column
date_candidates = [c for c in df.columns if "date" in c.lower()]
DATE_COL = date_candidates[0] if date_candidates else None
print("Detected date column:", DATE_COL)

# Unbalanced history: rows per month
if DATE_COL and DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    if df[DATE_COL].notna().any():
        monthly = df.groupby(df[DATE_COL].dt.to_period("M")).size()
        print("rows per month:\n", monthly)
        print("min month:", monthly.idxmin(), monthly.min(), "| max month:", monthly.idxmax(), monthly.max())

        # GSC-only early rows: does missingness change over time?
        min_date = df[DATE_COL].min()
        early = df[df[DATE_COL] <= min_date + pd.Timedelta(days=30)]
        later = df[df[DATE_COL] > min_date + pd.Timedelta(days=30)]
        print("\nmissing share, first 30 days:\n", early.isna().mean().sort_values(ascending=False).head())
        print("\nmissing share, rest of data:\n", later.isna().mean().sort_values(ascending=False).head())

        # Window overlaps: if you have two date-bounded sources/files, compare their ranges
        print("\noverall min/max date:", df[DATE_COL].min(), df[DATE_COL].max())
    else:
        print(f"'{DATE_COL}' exists but every value failed to parse as a date — check its format.")
else:
    print("No date-like column found — run print(df.columns.tolist()) and set DATE_COL manually.")
# **Data limits**


 #Unbalanced history.** Not every page has the same amount of history in this dataset. Newer pages have fewer months of data than older ones, so a page with a short history looks more volatile just because there's less data to average out, not because it actually performs less consistently.

 #GSC-only early rows.** The earliest records in this data come only from Google Search Console metrics (impressions, clicks, position), before other signals were being tracked. This means trends that reach back into that early period are comparing a narrower set of metrics than trends from later periods, so they aren't a fair apples-to-apples comparison.

 #Window overlaps.** This sample is a fixed window pulled from a larger warehouse. Rows near the edges of that window may be undercounted, since activity just before or after the cutoff isn't fully captured, and if this sample is ever combined with another cut of the same warehouse, overlapping rows could get double-counted.

#What this means for modeling:** conclusions drawn near the start of the data's history, or near the edges of the sampled window, are less reliable than conclusions from the stable middle of the range. The model shouldn't be expected to explain performance changes it never saw a complete, unbiased picture of.

Detected date column: None
No date-like column found — run print(df.columns.tolist()) and set DATE_COL manually.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.